# 🔋 Delhi DISCOM: Probabilistic Peak-Aware Load Forecasting & Procurement Copilot
### CodeSpark 2026 | Domain 4, PS 2
**Lead:** Varun Kumar (Machine Learning & Data Pipeline Lead)

---

## Executive Overview
Delhi DISCOM faces intense load swings between winter (~2,000 MW) and summer heatwaves (~8,300 MW).
This notebook provides the complete reproducible technical walkthrough:
1. **Zero-Leakage Data Preprocessing & Feature Engineering** (IMD Weather Forecast, CDH, 72h Thermal Inertia)
2. **Dual Benchmark Baselines** (Seasonal-Naive 168h and Yesterday-Naive 24h)
3. **Multi-Horizon Quantile Regression** (LightGBM P10, P50, P90)
4. **Conformalized Quantile Regression (CQR)** for 80% coverage calibration
5. **Cost-Derived Peak-Event Classifier** ($p^* \approx 0.222$ based on 3.5x DSM penalty)
6. **52-Window Rolling-Origin Walk-Forward Backtesting**
7. **TreeSHAP Explainability & Error Diagnostics**


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Load feature matrix and precomputed predictions
feat_df = pd.read_parquet('data/feature_matrix.parquet')
preds_df = pd.read_csv('outputs/predictions_with_bands.csv', index_col=0, parse_dates=True)
walkforward_df = pd.read_csv('outputs/walkforward_aggregate_metrics.csv', index_col=0)
print('Feature Matrix Shape:', feat_df.shape)
print('Validation Horizon:', len(preds_df), 'hourly time steps')


## 1. Reconciled Benchmark Performance (52-Window Walk-Forward)
Our 52-window rolling-origin evaluation strictly tests the model on sequential unseen weeks with zero future data leakage.

In [2]:
print('=== 52-WEEK WALK-FORWARD CROSS-VALIDATION SUMMARY ===')
print(walkforward_df[['mean', 'std', 'min', 'max']].round(3))


## 2. Visualizing Probabilistic P10 - P50 - P90 Day-Ahead Prediction Bands
The shaded band represents the P10-P90 prediction interval. Red dots highlight hours flagged by the cost-optimal peak classifier ($p \ge 0.222$).

In [3]:
sample = preds_df.iloc[:24*14]  # 2 weeks
plt.figure(figsize=(15, 6))
plt.plot(sample.index, sample['actual_mw'], color='black', linewidth=1.5, label='Actual Load (MW)')
plt.plot(sample.index, sample['pred_q50_h24'], color='#2b6cb0', linewidth=1.5, label='P50 Day-Ahead Forecast')
plt.fill_between(sample.index, sample['pred_q10_h24'], sample['pred_q90_h24'], color='#63b3ed', alpha=0.3, label='P10 - P90 Conformalized Confidence Band')

peak_sample = sample[sample['pred_peak_flag'] == 1]
if len(peak_sample) > 0:
    plt.scatter(peak_sample.index, peak_sample['actual_mw'], color='red', s=40, zorder=5, label='Peak Surge Alert (Cost Thresh)')

plt.title('Delhi DISCOM: Day-Ahead Probabilistic Load Forecast (P10/P50/P90)', fontsize=13, fontweight='bold')
plt.ylabel('Load (MW)')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 3. TreeSHAP Global Feature Explainability
TreeSHAP reveals the exact physical drivers of the load forecast, highlighting the dominance of **72h Thermal Inertia** and **Cooling Degree Hours (CDH)**.

In [4]:
from IPython.display import Image, display
print('TreeSHAP Global Attribution:')
display(Image(filename='outputs/shap_summary.png'))
print('Thermal Inertia Dependence (72h Rolling Temp vs MW Impact):')
display(Image(filename='outputs/shap_dependence_72h.png'))


## 4. Season x Hour Error Diagnostics & Top Missed Peak Days

In [5]:
display(Image(filename='outputs/error_heatmap.png'))
print('\nTop 5 Missed Peak Days (Known Failure Modes):')
missed_peaks = pd.read_csv('outputs/top_5_missed_peak_days.csv')
print(missed_peaks.to_string(index=False))


## 5. Rupee Financial Procurement Backtest (3.5x DSM Penalty)

In [6]:
rupee_df = pd.read_csv('outputs/rupee_procurement_impact.csv')
print(rupee_df.to_string(index=False))
